In [3]:
import os
import requests
import pandas as pd
from transformers import AutoTokenizer, AutoModelForSequenceClassification, pipeline
import matplotlib.pyplot as plt
from datetime import datetime, timedelta
import re

# === Configuration ===
NEWS_API_KEY = 'e3762f837b5d4677a8fc78db2fdc0d2f'  # Replace with your NewsAPI key
date_str = datetime.today().strftime('%Y-%m-%d')
sentiment_dir = os.path.join("sentiment", date_str)
chart_dir = os.path.join("charts", date_str)
os.makedirs(sentiment_dir, exist_ok=True)
os.makedirs(chart_dir, exist_ok=True)

# === Stock symbol to smart query map ===
stock_queries = {
    'AAPL': 'Apple Inc',
    'MSFT': 'Microsoft Corporation',
    'GOOGL': 'Alphabet Inc',
    'AMZN': 'Amazon.com Inc',
    'NVDA': 'NVIDIA Corporation',
    'META': 'Meta Platforms Inc',
    'TSLA': 'Tesla Inc',
    'BRK-B': 'Berkshire Hathaway Inc',
    'UNH': 'UnitedHealth Group Incorporated',
    'JPM': 'JPMorgan Chase & Co'
}

# === Load FinBERT Sentiment Model ===
print("Loading FinBERT model...")
tokenizer = AutoTokenizer.from_pretrained("ProsusAI/finbert")
model = AutoModelForSequenceClassification.from_pretrained("ProsusAI/finbert")
sentiment_pipeline = pipeline("sentiment-analysis", model=model, tokenizer=tokenizer)

# === Helper: Clean text ===
def clean_text(text):
    if not text:
        return ""
    return re.sub(r'\s+', ' ', text).strip()

# === Loop through each stock ===
for symbol, query in stock_queries.items():
    print(f"\nFetching news for {symbol} using query: {query}")
    
    all_articles = []

    # Loop over the past 7 days
    for i in range(7):
        day = datetime.today() - timedelta(days=i)
        day_str = day.strftime('%Y-%m-%d')

        url = (
            f"https://newsapi.org/v2/everything?q={query}&from={day_str}&to={day_str}"
            f"&sortBy=publishedAt&pageSize=14&apiKey={NEWS_API_KEY}&language=en"
        )

        try:
            response = requests.get(url)
            response.raise_for_status()
            articles = response.json().get("articles", [])
        except Exception as e:
            print(f"Error fetching news for {symbol} on {day_str}: {e}")
            continue

        for article in articles:
            if not article.get("publishedAt"):
                continue
            title = clean_text(article.get("title", ""))
            description = clean_text(article.get("description", ""))
            if title:
                all_articles.append({
                    "date": article["publishedAt"][:10],
                    "title": title,
                    "description": description
                })

    if not all_articles:
        print(f"No valid news articles found for {symbol}.")
        continue

    # Run sentiment analysis
    texts = [f"{a['title']}. {a['description']}" for a in all_articles]
    try:
        results = sentiment_pipeline(texts)
    except Exception as e:
        print(f"Sentiment analysis failed for {symbol}: {e}")
        continue

    # Build DataFrame
    df = pd.DataFrame(all_articles)
    df["sentiment"] = [r["label"] for r in results]
    df["confidence"] = [r["score"] for r in results]

    # Save CSV
    csv_path = os.path.join(sentiment_dir, f"{symbol}_sentiment.csv")
    df.to_csv(csv_path, index=False)
    print(f"Saved sentiment CSV to {csv_path}")

    # Save Chart
    plt.figure(figsize=(6, 4))
    df["sentiment"].value_counts().plot(kind='bar', color=["green", "red", "gray"])
    plt.title(f"Sentiment for {symbol} News (Last 7 Days, 14/Day)")
    plt.xlabel("Sentiment")
    plt.ylabel("Number of Articles")
    plt.xticks(rotation=0)
    plt.tight_layout()

    chart_path = os.path.join(chart_dir, f"{symbol}_chart.png")
    plt.savefig(chart_path)
    plt.close()
    print(f"Saved sentiment chart to {chart_path}")

print(f"\n✅ All sentiment CSVs saved in: {sentiment_dir}")
print(f"✅ All sentiment charts saved in: {chart_dir}")


Loading FinBERT model...


Device set to use cpu



Fetching news for AAPL using query: Apple Inc
Saved sentiment CSV to sentiment/2025-07-03/AAPL_sentiment.csv
Saved sentiment chart to charts/2025-07-03/AAPL_chart.png

Fetching news for MSFT using query: Microsoft Corporation
Saved sentiment CSV to sentiment/2025-07-03/MSFT_sentiment.csv
Saved sentiment chart to charts/2025-07-03/MSFT_chart.png

Fetching news for GOOGL using query: Alphabet Inc
Saved sentiment CSV to sentiment/2025-07-03/GOOGL_sentiment.csv
Saved sentiment chart to charts/2025-07-03/GOOGL_chart.png

Fetching news for AMZN using query: Amazon.com Inc
Saved sentiment CSV to sentiment/2025-07-03/AMZN_sentiment.csv
Saved sentiment chart to charts/2025-07-03/AMZN_chart.png

Fetching news for NVDA using query: NVIDIA Corporation
Saved sentiment CSV to sentiment/2025-07-03/NVDA_sentiment.csv
Saved sentiment chart to charts/2025-07-03/NVDA_chart.png

Fetching news for META using query: Meta Platforms Inc
Saved sentiment CSV to sentiment/2025-07-03/META_sentiment.csv
Saved sen